# 03 · QC instance và ba checkpoint instance

Xuất `COCO 1.0`. Một object vật lý = một mask. Tự vẽ object Medium đầu trước gợi ý tự động và ghi quy tắc vào `REPORT.md`; gợi ý không thay quyết định của bạn. COCO `annotation_id` không phải mã object bền vững qua hai lần export.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/manifest.json").is_file()), None)
assert root is not None, "Mở notebook từ thư mục repo Day 5 (hoặc thư mục notebooks/)."
sys.path.insert(0, str(root / "scripts"))
from inspect_submissions import task_registry, expected_for, inspect_task
tasks = task_registry(root)
exports = root / "submissions"
print("Repo:", root)
print("Thư mục export:", exports)


In [ ]:
instance_names = [name for name, info in tasks.items() if info["type"] == "instance"]
for name in instance_names:
    result = inspect_task(name, exports / f"{name}.zip", root)
    status = "LỖI" if result["errors"] else ("CHƯA XUẤT" if not Path(result["file"]).exists() else "OK")
    print("\n", name, "·", status)
    print("  số annotation:", result["details"].get("annotation_count", "—"))
    print("  kiểu mask:", result["details"].get("segmentation_kinds", {}))
    for note in result["errors"]: print("  SỬA:", note)
    for note in result["warnings"]: print("  KIỂM:", note)


## Đếm theo ảnh và class

Bảng này là số mask **bạn đã nộp**, không phải số object đúng. So lại trực quan với từng ảnh trong CVAT để tìm thiếu/thừa, gộp/tách sai, vật bị che vẫn là một object.

In [ ]:
task_name = "medium_instance"  # đổi thành cp1_holes, cp2_slice hoặc cp5_occlusion
result = inspect_task(task_name, exports / f"{task_name}.zip", root)
for key, count in result["details"].get("counts_by_image_class", {}).items():
    print(f"{key}: {count}")
if not result["details"].get("counts_by_image_class"): print("Chưa có object để đếm; kiểm ZIP và format.")


## Ca cần phán đoán

- `cp1_holes`: theo quy tắc task, kính/lỗ nằm trong mask, không tự khoét.
- `cp2_slice`: hai xe cùng lớp sát nhau vẫn là hai instance.
- `cp5_occlusion`: vật bị che thành hai phần nhìn thấy vẫn là một instance.
- Nếu class sai hoặc mask ăn nền, sửa trong CVAT, Save, export lại ZIP.